Generate text vs generate images: two API calls

In [9]:
import google.generativeai as genai
import os
from google.colab import userdata



API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)
prompt = 'A lake view from a banayan tree'
full_instruction = f'Describe this scence vividly in 3 sentences. {prompt}'


print("Text Generation")
model = genai.GenerativeModel('gemini-2.5-flash', generation_config={'max_output_tokens':150, 'temperature':0.7})

response = model.generate_content(full_instruction)
print(f'Response: {response.text}')

usage = response.usage_metadata
print(f"Prompt (Input) Tokens: {usage.prompt_token_count}")
print(f"Candidates (Output) Tokens: {usage.candidates_token_count}")
print(f"Total Tokens Used: {usage.total_token_count}")



print("Image Generation")
print("\n" + "=" * 60)
print("IMAGE GENERATION (Diffusion)")
print("=" * 60)

image_response = client.images.generate(
    model="dall-e-3",
    prompt=prompt,
    size="1024x1024",
    quality="standard",
    n=1
)

print(f"Prompt: {prompt}")
print(f"Image URL: {image_response.data[0].url}")
print(f"Revised prompt: {image_response.data[0].revised_prompt}")

SyntaxError: unterminated string literal (detected at line 28) (3272338057.py, line 28)

### Listing available models

In [ ]:
print("Available Models:")
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(f"  Model Name: {m.name}, Supported Methods: {m.supported_generation_methods}")

Diffusion vs GAN: speed and quality trade-off


In [ ]:
!pip install diffusers torch transformers accelerate

In [4]:
import torch
import time
from huggingface_hub import login
from google.colab import userdata

print(f'Standard Diffusion')
from diffusers import StableDiffusionPipeline

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
else:
    print("Please add your Hugging Face token")

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype = torch.float16
).to('cuda')

prompt = 'A photorealistic mountain view from bank of river'
start = time.time()
image_50 = pipe(prompt, num_inference_steps=50).images[0]
time_50 = time.time() - start

print(f'50 Steps: {time_50:.1f}s')
image_50.save('diffusion_50_img.png')

Standard Diffusion


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

50 Steps: 7.4s


In [3]:
import torch
import time
from huggingface_hub import login
from google.colab import userdata

print(f'Standard Diffusion: Reduced Denoising Steps (10)')
from diffusers import StableDiffusionPipeline

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
else:
    print("Please add your Hugging Face token")

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype = torch.float16
).to('cuda')


prompt = 'A photorealistic mountain view from bank of river'
image_10 = pipe(prompt,num_inference_steps=10).images[0]
time_10 = time.time() - start

print(f'10 Steps: {time_10:.1f}s')
image_10.save('diffusion_10_img.png')

Standard Diffusion: Reduced Denoising Steps (10)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

10 Steps: 310.1s


Compare the saved Images and Quality